
# Roxy notebook example: Sequence-order descriptors

This notebook is a **reference implementation example** for the **sequence-order descriptor family** in Roxy.

Sequence-order descriptors go beyond global composition by trying to capture **how residues or residue groups are arranged along the sequence**.

## Covered outputs

This notebook implements examples of sequence-order descriptors such as:

- transitions between residue groups
- adjacency enrichment
- mean spacing between residues of interest
- mean spacing between grouped residues
- local clustering proxies
- same-group adjacency fractions
- charged / hydrophobic neighborhood tendencies
- simple sequence-order coupling summaries
- class-style implementation for later migration into Roxy

The main goal is to provide a **clean teaching implementation** that can later become a real `order.py` module in Roxy.


In [1]:

from collections import Counter, defaultdict
from itertools import combinations

import numpy as np
import pandas as pd


## Demo dataset

In [2]:

df_demo = pd.DataFrame(
    {
        "sequence_id": [
            "ord_1",
            "ord_2",
            "ord_3",
            "ord_4",
            "ord_5",
            "ord_6",
        ],
        "sequence": [
            "MKWVTFISLLFLFSSAYSRGVFRR",
            "GGGGGGGGGGGGGGG",
            "KRRKRRKRRKRRDDDDEE",
            "ACDEFGHIKLMNPQRSTVWY",
            "PPPPGSSSSSTTTTNNQQQ",
            "MSTNPKPQRITLKDGNKVELV",
        ],
        "label": ["A", "B", "A", "B", "A", "B"],
    }
)

df_demo


,sequence_id,sequence,label
0,ord_1,MKWVTFISLLFLFSSAYSRGVFRR,A
1,ord_2,GGGGGGGGGGGGGGG,B
2,ord_3,KRRKRRKRRKRRDDDDEE,A
3,ord_4,ACDEFGHIKLMNPQRSTVWY,B
4,ord_5,PPPPGSSSSSTTTTNNQQQ,A
5,ord_6,MSTNPKPQRITLKDGNKVELV,B


## Constants

In [3]:

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

AA_GROUPS = {
    "positive": set("KRH"),
    "negative": set("DE"),
    "charged": set("KRHDE"),
    "polar": set("STNQCYWHKRDE"),
    "nonpolar": set("AVLIMFGP"),
    "aromatic": set("FWYH"),
    "aliphatic": set("AVLIM"),
    "hydrophobic": set("AVLIMFWCY"),
    "hydrophilic": set("RNDQEHKST"),
    "disorder_promoting": set("ARGQSEPK"),
    "order_promoting": set("CWYFILNV"),
}


## Helper functions

In [4]:

def clean_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper().replace("*", "")
    return "".join([aa for aa in seq if aa in STANDARD_AA])


def binary_membership_vector(seq: str, aa_group) -> np.ndarray:
    return np.array([1 if aa in aa_group else 0 for aa in seq], dtype=int)


def adjacent_pairs(seq: str):
    if len(seq) < 2:
        return []
    return [(seq[i], seq[i + 1]) for i in range(len(seq) - 1)]


def same_group_adjacency_fraction(seq: str, aa_group) -> float:
    if len(seq) < 2:
        return np.nan
    pairs = adjacent_pairs(seq)
    same = sum((a in aa_group) and (b in aa_group) for a, b in pairs)
    return same / len(pairs)


def cross_group_transition_fraction(seq: str, group_a, group_b) -> float:
    if len(seq) < 2:
        return np.nan
    pairs = adjacent_pairs(seq)
    hits = 0
    for a, b in pairs:
        cond1 = (a in group_a and b in group_b)
        cond2 = (a in group_b and b in group_a)
        if cond1 or cond2:
            hits += 1
    return hits / len(pairs)


def positions_of_group(seq: str, aa_group):
    return [i for i, aa in enumerate(seq) if aa in aa_group]


def mean_spacing_positions(positions) -> float:
    if len(positions) < 2:
        return np.nan
    return float(np.mean(np.diff(positions)))


def normalized_mean_spacing(seq: str, aa_group) -> float:
    if len(seq) == 0:
        return np.nan
    positions = positions_of_group(seq, aa_group)
    spacing = mean_spacing_positions(positions)
    if np.isnan(spacing):
        return np.nan
    return spacing / len(seq)


def local_cluster_fraction(seq: str, aa_group, window: int = 3) -> float:
    if len(seq) < window:
        return np.nan
    hits = 0
    total = len(seq) - window + 1
    for i in range(total):
        w = seq[i:i+window]
        if sum(aa in aa_group for aa in w) >= 2:
            hits += 1
    return hits / total


def lag_coupling(seq: str, aa_group, lag: int = 1) -> float:
    if len(seq) <= lag:
        return np.nan
    x = binary_membership_vector(seq, aa_group)
    return float(np.mean(x[:-lag] * x[lag:]))


def alternating_transition_fraction(seq: str, group_a, group_b) -> float:
    if len(seq) < 2:
        return np.nan
    pairs = adjacent_pairs(seq)
    total = len(pairs)
    hits = 0
    for a, b in pairs:
        if (a in group_a and b in group_b) or (a in group_b and b in group_a):
            hits += 1
    return hits / total


def adjacency_enrichment(seq: str, aa_group) -> float:
    if len(seq) < 2:
        return np.nan
    p = np.mean([aa in aa_group for aa in seq])
    observed = same_group_adjacency_fraction(seq, aa_group)
    expected = p * p
    if expected == 0:
        return np.nan
    return observed / expected


## Core descriptor function

In [5]:

def sequence_order_descriptors(seq: str) -> dict:
    seq = clean_sequence(seq)
    out = {
        "ord_length": len(seq),
        "ord_valid_residue_count": len(seq),
    }

    if len(seq) == 0:
        return out

    groups_to_track = [
        "charged",
        "hydrophobic",
        "polar",
        "aromatic",
        "disorder_promoting",
        "order_promoting",
    ]

    for name in groups_to_track:
        group = AA_GROUPS[name]
        out[f"ord_{name}_same_adj_frac"] = same_group_adjacency_fraction(seq, group)
        out[f"ord_{name}_mean_spacing_norm"] = normalized_mean_spacing(seq, group)
        out[f"ord_{name}_cluster_frac_w3"] = local_cluster_fraction(seq, group, window=3)
        out[f"ord_{name}_lag1_coupling"] = lag_coupling(seq, group, lag=1)
        out[f"ord_{name}_lag2_coupling"] = lag_coupling(seq, group, lag=2)
        out[f"ord_{name}_adj_enrichment"] = adjacency_enrichment(seq, group)

    # Cross-group transition descriptors
    out["ord_charged_hydrophobic_transition_frac"] = cross_group_transition_fraction(
        seq, AA_GROUPS["charged"], AA_GROUPS["hydrophobic"]
    )
    out["ord_polar_nonpolar_transition_frac"] = cross_group_transition_fraction(
        seq, AA_GROUPS["polar"], AA_GROUPS["nonpolar"]
    )
    out["ord_disorder_order_transition_frac"] = cross_group_transition_fraction(
        seq, AA_GROUPS["disorder_promoting"], AA_GROUPS["order_promoting"]
    )
    out["ord_positive_negative_transition_frac"] = cross_group_transition_fraction(
        seq, AA_GROUPS["positive"], AA_GROUPS["negative"]
    )

    return out


## Functional usage on one sequence

In [6]:

example = sequence_order_descriptors(df_demo.loc[0, "sequence"])
list(example.items())[:16]


[('ord_length', 24),
 ('ord_valid_residue_count', 24),
 ('ord_charged_same_adj_frac', 0.043478260869565216),
 ('ord_charged_mean_spacing_norm', 0.3055555555555555),
 ('ord_charged_cluster_frac_w3', 0.045454545454545456),
 ('ord_charged_lag1_coupling', 0.043478260869565216),
 ('ord_charged_lag2_coupling', 0.0),
 ('ord_charged_adj_enrichment', np.float64(1.565217391304348)),
 ('ord_hydrophobic_same_adj_frac', 0.34782608695652173),
 ('ord_hydrophobic_mean_spacing_norm', 0.0673076923076923),
 ('ord_hydrophobic_cluster_frac_w3', 0.7272727272727273),
 ('ord_hydrophobic_lag1_coupling', 0.34782608695652173),
 ('ord_hydrophobic_lag2_coupling', 0.2727272727272727),
 ('ord_hydrophobic_adj_enrichment', np.float64(1.0221827861579411)),
 ('ord_polar_same_adj_frac', 0.21739130434782608),
 ('ord_polar_mean_spacing_norm', 0.09166666666666667)]

## Apply sequence-order descriptors to the full dataset

In [7]:

df_ord = pd.concat(
    [
        df_demo,
        df_demo["sequence"].apply(sequence_order_descriptors).apply(pd.Series),
    ],
    axis=1,
)

df_ord.head()


,sequence_id,sequence,label,ord_length,ord_valid_residue_count,ord_charged_same_adj_frac,ord_charged_mean_spacing_norm,ord_charged_cluster_frac_w3,ord_charged_lag1_coupling,ord_charged_lag2_coupling,...,ord_order_promoting_same_adj_frac,ord_order_promoting_mean_spacing_norm,ord_order_promoting_cluster_frac_w3,ord_order_promoting_lag1_coupling,ord_order_promoting_lag2_coupling,ord_order_promoting_adj_enrichment,ord_charged_hydrophobic_transition_frac,ord_polar_nonpolar_transition_frac,ord_disorder_order_transition_frac,ord_positive_negative_transition_frac
0,ord_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24.0,24.0,0.043478,0.305556,0.045455,0.043478,0.000000,...,0.304348,0.071970,0.590909,0.304348,0.227273,1.217391,0.130435,0.478261,0.347826,0.000000
1,ord_2,GGGGGGGGGGGGGGG,B,15.0,15.0,0.000000,NaN,0.000000,0.000000,0.000000,...,0.000000,NaN,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.000000,0.000000
2,ord_3,KRRKRRKRRKRRDDDDEE,A,18.0,18.0,1.000000,0.055556,1.000000,1.000000,1.000000,...,0.000000,NaN,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.000000,0.058824
3,ord_4,ACDEFGHIKLMNPQRSTVWY,B,20.0,20.0,0.052632,0.150000,0.166667,0.052632,0.055556,...,0.105263,0.128571,0.222222,0.105263,0.166667,0.657895,0.263158,0.578947,0.315789,0.000000
4,ord_5,PPPPGSSSSSTTTTNNQQQ,A,19.0,19.0,0.000000,NaN,0.000000,0.000000,0.000000,...,0.055556,0.052632,0.117647,0.055556,0.000000,5.013889,0.000000,0.055556,0.055556,0.000000


## Inspect order descriptor columns

In [8]:

ord_cols = [c for c in df_ord.columns if c.startswith("ord_") and c not in {"ord_length", "ord_valid_residue_count"}]
len(ord_cols), ord_cols[:15]


(40,
 ['ord_charged_same_adj_frac',
  'ord_charged_mean_spacing_norm',
  'ord_charged_cluster_frac_w3',
  'ord_charged_lag1_coupling',
  'ord_charged_lag2_coupling',
  'ord_charged_adj_enrichment',
  'ord_hydrophobic_same_adj_frac',
  'ord_hydrophobic_mean_spacing_norm',
  'ord_hydrophobic_cluster_frac_w3',
  'ord_hydrophobic_lag1_coupling',
  'ord_hydrophobic_lag2_coupling',
  'ord_hydrophobic_adj_enrichment',
  'ord_polar_same_adj_frac',
  'ord_polar_mean_spacing_norm',
  'ord_polar_cluster_frac_w3'])

In [9]:

df_ord[
    [
        "sequence_id",
        "ord_charged_same_adj_frac",
        "ord_hydrophobic_same_adj_frac",
        "ord_charged_mean_spacing_norm",
        "ord_hydrophobic_cluster_frac_w3",
        "ord_polar_nonpolar_transition_frac",
        "ord_disorder_order_transition_frac",
    ]
]


,sequence_id,ord_charged_same_adj_frac,ord_hydrophobic_same_adj_frac,ord_charged_mean_spacing_norm,ord_hydrophobic_cluster_frac_w3,ord_polar_nonpolar_transition_frac,ord_disorder_order_transition_frac
0,ord_1,0.043478,0.347826,0.305556,0.727273,0.478261,0.347826
1,ord_2,0.000000,0.000000,NaN,0.000000,0.000000,0.000000
2,ord_3,1.000000,0.000000,0.055556,0.000000,0.000000,0.000000
3,ord_4,0.052632,0.210526,0.150000,0.333333,0.578947,0.315789
4,ord_5,0.000000,0.000000,NaN,0.000000,0.055556,0.055556
5,ord_6,0.050000,0.050000,0.123810,0.157895,0.700000,0.400000


## Dataset-level summary

In [10]:

ord_summary = (
    df_ord[ord_cols]
    .mean(axis=0, numeric_only=True)
    .sort_values(ascending=False)
    .rename("mean_value")
    .reset_index()
    .rename(columns={"index": "descriptor"})
)

ord_summary.head(15)


,descriptor,mean_value
0,ord_order_promoting_adj_enrichment,1.834794
1,ord_disorder_promoting_adj_enrichment,1.098879
2,ord_charged_adj_enrichment,1.004956
3,ord_polar_adj_enrichment,1.001578
4,ord_hydrophobic_adj_enrichment,0.891440
5,ord_aromatic_adj_enrichment,0.657895
6,ord_polar_cluster_frac_w3,0.603306
7,ord_disorder_promoting_cluster_frac_w3,0.576588
8,ord_disorder_promoting_same_adj_frac,0.482047
9,ord_disorder_promoting_lag1_coupling,0.482047


## Sanity checks

In [11]:

assert "ord_charged_same_adj_frac" in df_ord.columns
assert "ord_hydrophobic_mean_spacing_norm" in df_ord.columns
assert "ord_aromatic_cluster_frac_w3" in df_ord.columns
assert "ord_charged_lag1_coupling" in df_ord.columns
assert "ord_polar_nonpolar_transition_frac" in df_ord.columns
assert df_ord["ord_length"].min() > 0

print(f"Number of sequence-order descriptor columns: {len(ord_cols)}")
print("Sequence-order descriptor checks passed.")


Number of sequence-order descriptor columns: 40
Sequence-order descriptor checks passed.


## Class-style implementation closer to the real package

In [12]:

class SequenceOrderDescriptors:
    """Example class-style sequence-order implementation for later migration into Roxy."""

    def transform_sequence(self, seq: str) -> dict:
        return sequence_order_descriptors(seq)

    def transform(self, sequences) -> pd.DataFrame:
        return pd.DataFrame([self.transform_sequence(seq) for seq in sequences])


ord_transformer = SequenceOrderDescriptors()
ord_matrix = ord_transformer.transform(df_demo["sequence"].tolist())
ord_matrix.head()


,ord_length,ord_valid_residue_count,ord_charged_same_adj_frac,ord_charged_mean_spacing_norm,ord_charged_cluster_frac_w3,ord_charged_lag1_coupling,ord_charged_lag2_coupling,ord_charged_adj_enrichment,ord_hydrophobic_same_adj_frac,ord_hydrophobic_mean_spacing_norm,...,ord_order_promoting_same_adj_frac,ord_order_promoting_mean_spacing_norm,ord_order_promoting_cluster_frac_w3,ord_order_promoting_lag1_coupling,ord_order_promoting_lag2_coupling,ord_order_promoting_adj_enrichment,ord_charged_hydrophobic_transition_frac,ord_polar_nonpolar_transition_frac,ord_disorder_order_transition_frac,ord_positive_negative_transition_frac
0,24,24,0.043478,0.305556,0.045455,0.043478,0.000000,1.565217,0.347826,0.067308,...,0.304348,0.071970,0.590909,0.304348,0.227273,1.217391,0.130435,0.478261,0.347826,0.000000
1,15,15,0.000000,NaN,0.000000,0.000000,0.000000,NaN,0.000000,NaN,...,0.000000,NaN,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.000000,0.000000
2,18,18,1.000000,0.055556,1.000000,1.000000,1.000000,1.000000,0.000000,NaN,...,0.000000,NaN,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.000000,0.058824
3,20,20,0.052632,0.150000,0.166667,0.052632,0.055556,0.842105,0.210526,0.118750,...,0.105263,0.128571,0.222222,0.105263,0.166667,0.657895,0.263158,0.578947,0.315789,0.000000
4,19,19,0.000000,NaN,0.000000,0.000000,0.000000,NaN,0.000000,NaN,...,0.055556,0.052632,0.117647,0.055556,0.000000,5.013889,0.000000,0.055556,0.055556,0.000000


## Merge transformer output back to the dataset

In [13]:

df_ord_class = pd.concat([df_demo, ord_matrix], axis=1)
df_ord_class.head()


,sequence_id,sequence,label,ord_length,ord_valid_residue_count,ord_charged_same_adj_frac,ord_charged_mean_spacing_norm,ord_charged_cluster_frac_w3,ord_charged_lag1_coupling,ord_charged_lag2_coupling,...,ord_order_promoting_same_adj_frac,ord_order_promoting_mean_spacing_norm,ord_order_promoting_cluster_frac_w3,ord_order_promoting_lag1_coupling,ord_order_promoting_lag2_coupling,ord_order_promoting_adj_enrichment,ord_charged_hydrophobic_transition_frac,ord_polar_nonpolar_transition_frac,ord_disorder_order_transition_frac,ord_positive_negative_transition_frac
0,ord_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24,24,0.043478,0.305556,0.045455,0.043478,0.000000,...,0.304348,0.071970,0.590909,0.304348,0.227273,1.217391,0.130435,0.478261,0.347826,0.000000
1,ord_2,GGGGGGGGGGGGGGG,B,15,15,0.000000,NaN,0.000000,0.000000,0.000000,...,0.000000,NaN,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.000000,0.000000
2,ord_3,KRRKRRKRRKRRDDDDEE,A,18,18,1.000000,0.055556,1.000000,1.000000,1.000000,...,0.000000,NaN,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.000000,0.058824
3,ord_4,ACDEFGHIKLMNPQRSTVWY,B,20,20,0.052632,0.150000,0.166667,0.052632,0.055556,...,0.105263,0.128571,0.222222,0.105263,0.166667,0.657895,0.263158,0.578947,0.315789,0.000000
4,ord_5,PPPPGSSSSSTTTTNNQQQ,A,19,19,0.000000,NaN,0.000000,0.000000,0.000000,...,0.055556,0.052632,0.117647,0.055556,0.000000,5.013889,0.000000,0.055556,0.055556,0.000000



## Suggested next refactor into the package

A clean migration path into Roxy would be:

- move helper logic into `roxy/sequence/order.py`
- keep residue groups in `roxy/core/constants.py`
- expose a class such as `SequenceOrderDescriptors`
- allow configurable:
  - tracked groups
  - lag values
  - window sizes for clustering
  - which order descriptors to compute
- add tests for:
  - empty sequences
  - strongly repetitive sequences
  - alternating sequences
  - highly clustered charged sequences
  - lower-case input
  - invalid characters removed during cleaning


## Optional export

In [ ]:
# df_ord.to_csv("demo_sequence_order_descriptors.csv", index=False)
